# 04: HDD/CDD 先物・オプションのプライシング

## 不完全市場 (Incomplete Market) とは何か？

### 完全市場（株式デリバティブ）
株式オプションの場合、原資産（株）を売買してデルタヘッジができる。
→ **リスク中立測度 Q が一意** に決まる → 「正しい価格」が1つ存在

### 不完全市場（気候デリバティブ）
気温は**売買できない**。気温が上がったからといって「気温を売る」ことはできない。
→ デルタヘッジ不可能 → **Q が一意に決まらない** → 「正しい価格」が一意でない

### Market Price of Risk (λ)
Q が一意でないことを解決するために、**λ (Market Price of Risk)** を導入する:

| 測度 | SDE | 意味 |
|------|-----|------|
| P (実世界) | $dT = \kappa(\theta - T)dt + \sigma dW^P$ | 実際の気温ダイナミクス |
| Q (リスク中立) | $dT = [\kappa(\theta - T) - \lambda\sigma]dt + \sigma dW^Q$ | プライシング用 |

- **λ = 0**: リスクプレミアムなし（P = Q）
- **λ > 0**: 気温下方シフト → HDD先物が高く、CDD先物が低くなる
- **λ < 0**: 気温上方シフト → HDD先物が低く、CDD先物が高くなる

実務では λ は**市場価格から逆算** (implied λ) するか、リスク許容度に基づいて設定する。

In [ ]:
import sys
sys.path.insert(0, "../backend")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from collections import defaultdict

from app.services.noaa_client import generate_sample_data
from app.services.degree_days import (
    calculate_daily_degree_days,
    calculate_monthly_degree_days,
    DEFAULT_BASE_TEMP_C,
)
from app.models.temperature import fit_temperature_model
from app.models.pricing import burn_analysis, monte_carlo_pricing

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")

# 10年分のサンプルデータ
sample_data = generate_sample_data(
    start_date=date(2014, 1, 1),
    end_date=date(2023, 12, 31),
)

t = np.arange(len(sample_data), dtype=np.float64)
tavg = np.array([d.tavg for d in sample_data])

# モデルフィット
model_result = fit_temperature_model(t, tavg)
print(f"データ: {len(sample_data)} days, κ={model_result.ou.kappa:.4f}, σ={model_result.ou.sigma:.4f}")
print("Setup complete!")

## 1. Burn Analysis — ヒストリカルプライシング

過去10年の1月HDD累積値を使って、先物価格とオプション価格を計算する。

**先物価格** = 過去N年のHDD累積値の平均
**オプション価格** = 過去N年のペイオフの平均

In [ ]:
# 日次HDD/CDDを計算し、年×月で集計
daily_dd = calculate_daily_degree_days(sample_data)

# 1月のHDDを年ごとに集計
jan_hdd_by_year: dict[int, float] = defaultdict(float)
for dd in daily_dd:
    if dd.date.month == 1:
        jan_hdd_by_year[dd.date.year] += dd.hdd

# Burn Analysis 実行
burn_result = burn_analysis(
    historical_dd_by_year=dict(jan_hdd_by_year),
    strike=500.0,  # 行使価格 500 HDD
    discount_factor=0.99,  # 年率1%の割引
)

print("=" * 50)
print("Burn Analysis — January HDD Futures & Options")
print("=" * 50)
print(f"\n先物価格 (単純平均):   {burn_result.futures_price:.2f} HDD")
print(f"先物価格 (重み付き):   {burn_result.futures_price_weighted:.2f} HDD")
print(f"標準偏差:             {burn_result.std:.2f}")
print(f"95%信頼区間:          [{burn_result.confidence_interval[0]:.2f}, {burn_result.confidence_interval[1]:.2f}]")
print(f"\nコールオプション (K={500}): {burn_result.call_price:.2f}")
print(f"プットオプション (K={500}): {burn_result.put_price:.2f}")
print(f"\n年別HDD累積値:")
for y, v in zip(burn_result.years, burn_result.historical_values):
    print(f"  {y}: {v:.1f}")

## 2. Monte Carlo プライシング — λ の感応度分析

λ (Market Price of Risk) を変化させて、先物価格とオプション価格がどう変わるかを分析する。

**重要**: 不完全市場では λ の選択が価格を決定する。これは「正解がない」ことを意味し、
価格は市場参加者間の合意によって決まる。

In [ ]:
# λ を -1.0 ~ 1.0 まで変化させて、1月HDD先物・オプション価格を計算
lambdas = np.linspace(-1.0, 1.0, 21)
futures_prices = []
call_prices = []
put_prices = []

# 1月のシミュレーション (t_start: 1月1日に相当する位置)
# データの最終年の翌年1月を想定
t_start_jan = float(t[-1]) + 1.0  # データ末尾の翌日から
T_initial = float(model_result.seasonal.evaluate(np.array([t_start_jan]))[0])
strike = 500.0

for lam in lambdas:
    result = monte_carlo_pricing(
        seasonal=model_result.seasonal,
        ou=model_result.ou,
        t_start=t_start_jan,
        n_days=31,
        T_initial=T_initial,
        dd_type="hdd",
        strike=strike,
        discount_factor=0.99,
        lambda_risk=float(lam),
        n_paths=20000,
        seed=42,
    )
    futures_prices.append(result.futures_price)
    call_prices.append(result.call_price)
    put_prices.append(result.put_price)

# プロット
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 先物価格 vs λ
axes[0].plot(lambdas, futures_prices, "o-", color="navy", markersize=4)
axes[0].axhline(burn_result.futures_price, color="red", linestyle="--", alpha=0.7, label="Burn Analysis")
axes[0].axvline(0, color="gray", linestyle=":", alpha=0.5)
axes[0].set_xlabel("λ (Market Price of Risk)")
axes[0].set_ylabel("Futures Price (HDD)")
axes[0].set_title("HDD Futures Price vs λ")
axes[0].legend()

# コールオプション vs λ
axes[1].plot(lambdas, call_prices, "o-", color="green", markersize=4)
axes[1].axhline(burn_result.call_price, color="red", linestyle="--", alpha=0.7, label="Burn Analysis")
axes[1].axvline(0, color="gray", linestyle=":", alpha=0.5)
axes[1].set_xlabel("λ")
axes[1].set_ylabel("Call Price")
axes[1].set_title(f"Call Option (K={strike}) vs λ")
axes[1].legend()

# プットオプション vs λ
axes[2].plot(lambdas, put_prices, "o-", color="orangered", markersize=4)
axes[2].axhline(burn_result.put_price, color="red", linestyle="--", alpha=0.7, label="Burn Analysis")
axes[2].axvline(0, color="gray", linestyle=":", alpha=0.5)
axes[2].set_xlabel("λ")
axes[2].set_ylabel("Put Price")
axes[2].set_title(f"Put Option (K={strike}) vs λ")
axes[2].legend()

plt.tight_layout()
plt.show()

print("λ > 0 → 気温下方シフト → HDD増加 → 先物・コール価格↑、プット価格↓")
print("λ < 0 → 気温上方シフト → HDD減少 → 先物・コール価格↓、プット価格↑")
print(f"\n赤破線 = Burn Analysis価格 (λ不要のモデルフリーベンチマーク)")

## 3. パス数 vs 価格収束 — Monte Carlo の精度

Monte Carlo法の精度はパス数 N に依存する:
- 標準誤差 (SE) ∝ 1/√N
- N を4倍にすると SE は半分になる

十分なパス数を使わないと、価格の推定にノイズが乗る。

In [ ]:
# パス数を変えて先物価格と標準誤差を計算
path_counts = [100, 500, 1000, 2000, 5000, 10000, 20000, 50000]
convergence_prices = []
convergence_ses = []

for n in path_counts:
    result = monte_carlo_pricing(
        seasonal=model_result.seasonal,
        ou=model_result.ou,
        t_start=t_start_jan,
        n_days=31,
        T_initial=T_initial,
        dd_type="hdd",
        lambda_risk=0.0,
        n_paths=n,
        seed=42,
    )
    convergence_prices.append(result.futures_price)
    convergence_ses.append(result.futures_se)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 先物価格の収束
axes[0].plot(path_counts, convergence_prices, "o-", color="navy")
axes[0].axhline(convergence_prices[-1], color="red", linestyle="--", alpha=0.5, label=f"N=50000: {convergence_prices[-1]:.1f}")
axes[0].fill_between(path_counts,
                      [p - 2*s for p, s in zip(convergence_prices, convergence_ses)],
                      [p + 2*s for p, s in zip(convergence_prices, convergence_ses)],
                      alpha=0.2, color="navy")
axes[0].set_xscale("log")
axes[0].set_xlabel("Number of Paths (log scale)")
axes[0].set_ylabel("Futures Price (HDD)")
axes[0].set_title("Convergence of Futures Price")
axes[0].legend()

# 標準誤差の減少
axes[1].plot(path_counts, convergence_ses, "o-", color="orangered", label="Empirical SE")
# 理論的な 1/√N の線
se_base = convergence_ses[0] * np.sqrt(path_counts[0])
theoretical_se = [se_base / np.sqrt(n) for n in path_counts]
axes[1].plot(path_counts, theoretical_se, "--", color="gray", alpha=0.7, label="Theoretical ∝ 1/√N")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Number of Paths (log scale)")
axes[1].set_ylabel("Standard Error (log scale)")
axes[1].set_title("Standard Error vs Number of Paths")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"パス数 100:    SE = {convergence_ses[0]:.2f}")
print(f"パス数 10000:  SE = {convergence_ses[5]:.2f}")
print(f"パス数 50000:  SE = {convergence_ses[-1]:.2f}")
print(f"\n→ 実務では 10,000~50,000 パスが一般的")

## 4. Burn Analysis vs Monte Carlo — 手法比較

λ=0 の Monte Carlo は Burn Analysis と近い結果になるはずだが、完全には一致しない。

**違いの原因**:
- Burn Analysis: 過去の実績分布をそのまま使用
- Monte Carlo: OU過程モデルを介した分布（パラメトリック）
- モデルの仮定（正規残差、一定σ等）により分布の形が微妙に異なる

In [ ]:
# Monte Carlo (λ=0) の分布を取得
mc_result = monte_carlo_pricing(
    seasonal=model_result.seasonal,
    ou=model_result.ou,
    t_start=t_start_jan,
    n_days=31,
    T_initial=T_initial,
    dd_type="hdd",
    strike=strike,
    discount_factor=0.99,
    lambda_risk=0.0,
    n_paths=50000,
    seed=42,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 分布の比較
axes[0].hist(mc_result.distribution, bins=60, density=True, alpha=0.6,
             color="steelblue", edgecolor="white", label="Monte Carlo (λ=0)")
# Burn Analysisのヒストリカル値を縦線で表示
for i, v in enumerate(burn_result.historical_values):
    label = "Burn Analysis (historical)" if i == 0 else None
    axes[0].axvline(v, color="red", alpha=0.5, linewidth=1.5, label=label)
axes[0].axvline(burn_result.futures_price, color="red", linewidth=3, linestyle="--",
                label=f"Burn avg = {burn_result.futures_price:.1f}")
axes[0].axvline(mc_result.futures_price, color="navy", linewidth=3, linestyle="--",
                label=f"MC avg = {mc_result.futures_price:.1f}")
axes[0].set_xlabel("January Cumulative HDD")
axes[0].set_ylabel("Density")
axes[0].set_title("Distribution Comparison: MC vs Burn Analysis")
axes[0].legend(fontsize=9)

# 結果テーブル
comparison = pd.DataFrame({
    "Metric": ["Futures Price", "Call (K=500)", "Put (K=500)"],
    "Burn Analysis": [burn_result.futures_price, burn_result.call_price, burn_result.put_price],
    "Monte Carlo (λ=0)": [mc_result.futures_price, mc_result.call_price, mc_result.put_price],
})
comparison["Difference"] = comparison["Monte Carlo (λ=0)"] - comparison["Burn Analysis"]
comparison["Diff %"] = (comparison["Difference"] / comparison["Burn Analysis"] * 100).round(1)

# テーブルをプロット
axes[1].axis("off")
table = axes[1].table(
    cellText=comparison.round(2).values,
    colLabels=comparison.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
axes[1].set_title("Burn Analysis vs Monte Carlo Comparison", fontsize=13, pad=20)

plt.tight_layout()
plt.show()

## 5. まとめ

### このNotebookで学んだこと

| 概念 | 内容 |
|------|------|
| **不完全市場** | 気温は売買不可 → リスク中立測度が一意でない |
| **Market Price of Risk (λ)** | P→Q測度変換のパラメータ。市場参加者のリスク許容度を反映 |
| **Burn Analysis** | モデルフリーのベンチマーク。過去実績の単純統計量 |
| **Monte Carlo** | Q測度下でのパラメトリックプライシング。λで価格を制御 |
| **収束性** | SE ∝ 1/√N。精度と計算コストのトレードオフ |

### 手法の使い分け

| 場面 | 推奨手法 |
|------|---------|
| クイックな価格感覚 | Burn Analysis |
| 気候変動トレンドの反映 | Monte Carlo（トレンド項b≠0） |
| リスクプレミアムの反映 | Monte Carlo（λ≠0） |
| エキゾチックペイオフ | Monte Carlo |
| モデルリスクを避けたい | Burn Analysis |

### Step 4 では
これらの計算結果を **React + D3.js ダッシュボード** で可視化し、
λスライダーでインタラクティブにプライシングできるUIを構築する。